In [9]:
# As Kernel use CRISPR_Adelysis (Python 3.12.4)

import ast
from datetime import datetime
import random
from scripts.make_mastertables import count_no_of_gaps, find_DEL_closest_to_cut, find_min_and_max_gap_ranges, find_DEL_around_cut_range, size_of_del_around_the_cut, gap_span_analysed_window, return_shifted_DEL_range, return_category_of_DEL, return_cut_type_based_on_cut_categories, find_microhomology
from scripts.dict_functions import get_color_dic_35_violin
from scripts.plot_functions import save_stat, plot_sns_violin_and_print
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Get current date and time
now = datetime.now()
# Format the date to YYMMDD
date = now.strftime('%y%m%d')

In [15]:
grouped_df_norm = pd.read_csv('random_categories_grouped_norm_-2_50_0_250524.tsv', sep='\t')
grouped_df_norm[['Gap_category_1', 'Gap_category_2', 'Cut_type', 'End_participating_on_MH', 'DEL_orientation', 'End_3_5', 'Likely_cut_type']] = grouped_df_norm['Combination'].str.split('_', expand=True)
grouped_df_norm.head()

,Combination,sum,NormReads,Gap_category_1,Gap_category_2,Cut_type,End_participating_on_MH,DEL_orientation,End_3_5,Likely_cut_type
0,6_6_C_X_B_0_0,1778746,18.958379,6,6,C,X,B,0,0
1,12_12_O_X_D_0_0,1655614,17.646003,12,12,O,X,D,0,0
2,1_1_O_X_U_0_0,1567404,16.705836,1,1,O,X,U,0,0
3,6_6_C_C_B_0_0,1330373,14.179492,6,6,C,C,B,0,0
4,12_12_O_C_D_0_0,1262905,13.460399,12,12,O,C,D,0,0


In [18]:
df_end = grouped_df_norm
# Rename columne NormReads to %Reads
df_end.rename(columns={'NormReads': '%Reads'}, inplace=True)
# drop data without gap
# df_end = df_end[df_end['n_deleted_around_cut'] != 0]
# filter out deletions which are not around the cut
# df_end = df_end[df_end['Gap_around_cut'] == True]
# filter out deletionw which does not have MH around the cut site
df_end = df_end[df_end['End_participating_on_MH'] != 'X']
# Change values in End_3_5 to be 3', 5' and cannot_be_determined needed for plotting with color_dic
df_end['End_3_5'] = df_end['End_3_5'].replace({3:'3\'', 5:'5\'', 0:'eider 3\' or 5\''})
pivot = pd.pivot_table(df_end, index=['name', 'gRNA_seq'], columns=['End_3_5', 'blunt_cut_with_MH_possible'], values='%Reads', aggfunc='sum', fill_value=0)
df_end_samples = pivot.reset_index()
# Flattern the MultiIndex columns
df_end_samples.columns = ['_'.join(map(str, col)).strip() for col in df_end_samples.columns.values]
# Rename columns
df_end_samples.rename(columns={'name_': 'name', 'gRNA_seq_':'gRNA_seq', '3\'_True':'3\' possible blunt', '5\'_True':'5\' possible blunt', 'eider 3\' or 5\'_True':'eider 3\' or 5\' possible blunt', '3\'_False':'3\' NOT blunt', '5\'_False':'5\' NOT blunt', 'eider 3\' or 5\'_False':'eider 3\' or 5\' NOT blunt'}, inplace=True)

df_end_samples, list_of_events = check_if_all_events_are_present_after_pivot_reset_index(df_end_samples, ['3\' possible blunt', '5\' possible blunt', 'eider 3\' or 5\' possible blunt', '3\' NOT blunt', '5\' NOT blunt', 'eider 3\' or 5\' NOT blunt'], cut_maximum, cut_minimum)
df_end_samples = pd.melt(df_end_samples, id_vars=['name', 'gRNA_seq'], value_vars=list_of_events)
pivot_normalised = pd.crosstab(index=[df_end_samples['name'],df_end_samples['gRNA_seq']], columns=df_end_samples['variable'], values=df_end_samples['value'], aggfunc='sum', normalize='index')
pivot_normalised
pivot_NOTnorm = pd.crosstab(index=[df_end_samples['name'],df_end_samples['gRNA_seq']], columns=df_end_samples['variable'], values=df_end_samples['value'], aggfunc='sum')
df_end_samples_norm = pivot_normalised.reset_index()
df_end_samples_norm = pd.melt(df_end_samples_norm, id_vars=['name', 'gRNA_seq'], value_vars=list_of_events)
df_end_samples_NOTnorm = pivot_NOTnorm.reset_index()
df_end_samples_NOTnorm = pd.melt(df_end_samples_NOTnorm, id_vars=['name', 'gRNA_seq'], value_vars=list_of_events)

/tmp/ipykernel_46625/4277224469.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_end['End_3_5'] = df_end['End_3_5'].replace({3:'3\'', 5:'5\'', 0:'eider 3\' or 5\''})


KeyError: 'name'

In [14]:
#########################################################################################
# plot VIOLINPLOT
##########################################################################################
# NORMALISED
# Set charts properties for violinplot
title = 'Normalised ends participating on the repair'
ylabel = 'Normalised percentage of ends \n around cut events'
text_bellow_chart = f'Only DELs around the cut are plotted.\
                    \nNormalised to: SUM of all ends in individual sample = 100%'
name_to_save = f'05_01_DEL_end_summary_norm_violinplot'
color_dic = get_color_dic_35_violin()
# TODO split it to True/False for can be blunt cut with MH
plot = plot_sns_violin_and_print(data=grouped_df_norm, x='variable', y='value'\
    , number_of_columns=3\
    , show_datapoints=True, jitter=0.1, stripplot_offset=10, dodge=False\
    , show_hue_medians=False\
    , hue=None, hue_only_for_data_points=False \
    , hatches=False\
    , clip_right=True\
    , title=title, xtickt_rotation=-55, ylabel=ylabel\
    , order=['5\' NOT blunt','5\' possible blunt', '3\' NOT blunt', '3\' possible blunt', 'eider 3\' or 5\' NOT blunt', 'eider 3\' or 5\' possible blunt']\
    , plot_legend=False\
    , integrated_properties_dic=False, color_dic=color_dic\
    , text_bellow_chart=text_bellow_chart\
    , y_legend_coordinate=-0.45, yaxis_formater=1\
    , x_max=None\
    , change_edge_color=False\
    , name_to_save=name_to_save\
    , project_output_dir='./'\
    , data_source='random', output_format='png'\
    , dpi=200, y_min=0, y_max=1.02, show_plots=True)
stat_info = grouped_df_norm.groupby('variable').value.describe()
save_stat(stat_info, name_to_save, project_output_subdir)

The properties_dic is not a dictionary, and also the integrated dictionary is not going to be used.


KeyError: 'variable'